## ⚠️ Important Windows + CUDA Setup Prerequisites

Before you run this notebook, you **must** have the correct NVIDIA drivers and libraries installed on your Windows machine. The script will install `tensorflow==2.15.0`, which requires a specific setup.

**Your Setup Checklist:**
1.  **NVIDIA GPU:** Ensure you have a CUDA-capable NVIDIA GPU.
2.  **NVIDIA Driver:** Update to the latest Game Ready or Studio Driver from the NVIDIA website.
3.  **CUDA Toolkit:** You must install **CUDA 12.2**. (TF 2.15 is built with 12.2).
4.  **cuDNN SDK:** You must install **cuDNN v8.9** (for CUDA 12.x).

You can find the official tested build configurations on the [TensorFlow website](https://www.tensorflow.org/install/source_windows#tested_build_configurations).

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# The os.walk below is for Kaggle. You can ignore it for your local setup.
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        #print(os.path.join(dirname, filename))
        pass

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
# --- Cell 1: Ultimate Dependency Fix (Must be run first in your new env) ---

# 0. CRITICAL: Force Keras Backend
# This environment variable must be set BEFORE any Keras or TensorFlow imports.
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
# Set SM_FRAMEWORK as a secondary safety measure for segmentation-models
os.environ["SM_FRAMEWORK"] = "tf.keras"

# 1. Aggressive Uninstall (Remove lingering incompatible versions)
!pip uninstall -y keras tensorflow tf-keras-nightly tf-keras tensorflow-text tensorflow-decision-forests

# 2. Force Install Compatible Versions
print("Installing TensorFlow 2.15.0 and Keras 2.15.0...")
!pip install tensorflow==2.15.0 keras==2.15.0 --no-deps
!pip install Keras-Applications==1.0.8 --no-dependencies --no-cache-dir

# 3. Install specialized libraries
!pip install segmentation-models
# --- FIXED: Removed the deprecated tensorflow-addons line ---
!pip install keras-unet-collection 

# 4. Imports (Order matters here, as Keras must be loaded via TF)
import glob
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import tensorflow.keras.backend as K
import pandas as pd

# 5. CRITICAL Check and Fix
print(f"TensorFlow Version Check: {tf.__version__}")

# Clear V1 references if they persist (for keras-unet-collection)
try:
    del tf.compat.v1.image
    print("Cleaned up deprecated V1 references.")
except AttributeError:
    pass

# Import segmentation models (These should now resolve correctly)
try:
    # Use sm for the main module alias
    import segmentation_models as sm
    print("Segmentation Models (sm) imported successfully!")
    
except Exception as e:
    print(f"CRITICAL ERROR during segmentation_models import: {e}")
    
# Import Transformer-based models
try:
    from keras_unet_collection import models
    print("Keras-UNet-Collection imported successfully!")
except Exception as e:
    print(f"CRITICAL ERROR during keras_unet_collection import: {e}")

# --- Global Configuration (Keep these the same) ---
IMAGE_SIZE = (256, 256) 
INPUT_SHAPE = IMAGE_SIZE + (3,) 
BACKBONE = 'resnet34' 
BATCH_SIZE = 16 
N_CLASSES = 1

Installing TensorFlow 2.15.0 and Keras 2.15.0...


ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow==2.15.0


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# --- Cell 2: Custom Metrics and Loss ---

SMOOTH = 1e-6 

# 1. DICE COEFFICIENT (Metric)
def dice_coef(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + SMOOTH) / (K.sum(y_true_f) + K.sum(y_pred_f) + SMOOTH)

# 2. JACCARD INDEX (IoU) (Metric)
def iou_score(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + SMOOTH) / (union + SMOOTH)

# 3. SENSITIVITY (Recall) (Metric)
def sensitivity(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    return true_positives / (possible_positives + K.epsilon())

# 4. SPECIFICITY (Metric)
def specificity(y_true, y_pred):
    true_negatives = K.sum(K.round(K.clip((1 - y_true) * (1 - y_pred), 0, 1)))
    possible_negatives = K.sum(K.round(K.clip(1 - y_true, 0, 1)))
    return true_negatives / (possible_negatives + K.epsilon())

# HYBRID LOSS FUNCTION (Dice Loss + Binary Cross-Entropy)
def hybrid_loss(y_true, y_pred, alpha=0.5):
    """L_total = alpha * L_bce + (1 - alpha) * L_dice"""
    bce_loss = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice_loss = 1 - dice_coef(y_true, y_pred)
    return alpha * bce_loss + (1 - alpha) * dice_loss

METRICS = [iou_score, dice_coef, sensitivity, specificity, 'accuracy']

In [ ]:
# --- Cell 3: Data Preparation and Generator ---

# --- Define Paths (Relative to your Notebook file) ---
# **FIXED**: Using os.path.join for Windows compatibility
FRAMES_DIR = os.path.join('archive', 'frames')
MASKS_DIR = os.path.join('archive', 'masks')

# --- Load and Sort Files ---
# **FIXED**: Corrected glob to search inside FRAMES_DIR
all_frames = sorted(glob.glob(os.path.join(FRAMES_DIR, '*.png')))
all_masks = sorted(glob.glob(os.path.join(MASKS_DIR, '*.png')))

if len(all_frames) == 0 or len(all_frames) != len(all_masks):
    print(f"FATAL ERROR: Data files not found or mismatched counts.")
    print(f"Checked for frames in: {os.path.abspath(FRAMES_DIR)}")
    print(f"Checked for masks in: {os.path.abspath(MASKS_DIR)}")
else:
    print(f"Total paired samples found: {len(all_frames)}")

# --- Split Data ---
# 80% Train, 20% Validation
train_frames, val_frames, train_masks, val_masks = train_test_split(
    all_frames, all_masks, test_size=0.2, random_state=42
)
print(f"Train samples: {len(train_frames)}, Validation samples: {len(val_masks)}")


# --- Custom Data Generator (Keras Sequence) ---
class SegDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, frame_paths, mask_paths, batch_size, image_size, preprocess_input):
        self.frame_paths = frame_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.image_size = image_size
        self.preprocess_input = preprocess_input
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.frame_paths) / self.batch_size))

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.frame_paths))
        np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]

        frame_batch_paths = [self.frame_paths[k] for k in indexes]
        mask_batch_paths = [self.mask_paths[k] for k in indexes]

        X, y = self.__data_generation(frame_batch_paths, mask_batch_paths)
        return X, y

    def __data_generation(self, frame_batch_paths, mask_batch_paths):
        X = np.empty((self.batch_size, *self.image_size, 3), dtype=np.float32)
        y = np.empty((self.batch_size, *self.image_size, N_CLASSES), dtype=np.float32)

        for i, (frame_path, mask_path) in enumerate(zip(frame_batch_paths, mask_batch_paths)):
            # Load, convert to array, and preprocess image
            img_arr = img_to_array(load_img(frame_path, target_size=self.image_size, color_mode='rgb'))
            X[i,] = self.preprocess_input(img_arr) 

            # Load mask (single channel) and scale to [0, 1]
            mask_arr = img_to_array(load_img(mask_path, target_size=self.image_size, color_mode='grayscale'))
            y[i,] = (mask_arr / 255.0).astype(np.float32)
        
        return X, y

# --- Create Generators ---
preprocess_input = sm.get_preprocessing(BACKBONE)

train_generator = SegDataGenerator(train_frames, train_masks, BATCH_SIZE, IMAGE_SIZE, preprocess_input)
val_generator = SegDataGenerator(val_frames, val_masks, BATCH_SIZE, IMAGE_SIZE, preprocess_input)

In [ ]:
# --- Cell 4: Training Setup ---

# --- COMMON CALLBACKS ---
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)


def train_and_evaluate_model(model_builder_func, model_name, epochs=50):
    print(f"\n--- Starting Training for Model: {model_name} ---")
    
    # **FIXED**: Using os.path.join for Windows-compatible save path
    checkpoint_path = os.path.join('.', f'{model_name}_best_model.weights.h5')
    
    # Checkpoint to save best weights
    model_checkpoint = ModelCheckpoint(checkpoint_path, 
                                          monitor='val_dice_coef', 
                                          mode='max', 
                                          save_best_only=True, 
                                          verbose=1)
    
    model = model_builder_func()
    
    # Train the model
    history = model.fit(
        train_generator,
        steps_per_epoch=len(train_generator),
        epochs=epochs,
        validation_data=val_generator,
        validation_steps=len(val_generator),
        callbacks=[early_stopping, lr_scheduler, model_checkpoint],
        verbose=1
    )

    # Load the best weights for final evaluation
    try:
        # **FIXED**: Use os.path.join to load weights
        model.load_weights(checkpoint_path)
        print(f"Loaded best weights for {model_name}.")
    except Exception as e:
        print(f"Could not load saved weights for {model_name}. Using last epoch. Error: {e}")

    # Evaluate the model
    # The metrics order is: [loss, iou_score, dice_coef, sensitivity, specificity, accuracy]
    results = model.evaluate(val_generator, steps=len(val_generator), verbose=0)
    
    metrics_map = {
        'Model': model_name,
        'Loss (Hybrid)': results[0],
        'IoU': results[1],
        'Dice Coefficient': results[2],
        'Sensitivity (Recall)': results[3],
        'Specificity': results[4],
        'Accuracy': results[5]
    }
    print(f"Evaluation Complete for {model_name}.")
    return metrics_map, model, history

# --- 4.1 CNN Model Builders (Now Correct) ---
# These functions rely on `segmentation_models` (sm) and will work
# now that the TF 2.15.0 environment is correctly set up.

def build_unet():
    model = sm.Unet(BACKBONE, encoder_weights='imagenet', classes=N_CLASSES, activation='sigmoid', input_shape=INPUT_SHAPE)
    model.compile(optimizer=Adam(learning_rate=1e-4), loss=hybrid_loss, metrics=METRICS)
    model.name = 'Unet_ResNet34'
    return model

def build_att_unet():
    model = sm.AttentionUnet(BACKBONE, encoder_weights='imagenet', classes=N_CLASSES, activation='sigmoid', input_shape=INPUT_SHAPE)
    model.compile(optimizer=Adam(learning_rate=1e-4), loss=hybrid_loss, metrics=METRICS)
    model.name = 'AttUnet_ResNet34'
    return model

def build_unet_plus():
    model = sm.UnetPlusPlus(BACKBONE, encoder_weights='imagenet', classes=N_CLASSES, activation='sigmoid', input_shape=INPUT_SHAPE)
    model.compile(optimizer=Adam(learning_rate=1e-4), loss=hybrid_loss, metrics=METRICS)
    model.name = 'Unet++_ResNet34'
    return model

# --- 4.2 Transformer Model Builders ---
# These models rely on `keras_unet_collection` (models) and will also
# work with the TF 2.15.0 setup.

def build_trans_unet():
    # Hybrid CNN-ViT Model
    model = models.transunet_2d(
        INPUT_SHAPE, 
        filter_num=[64, 128, 256, 512, 1024], 
        n_labels=N_CLASSES, 
        stack_num_up=2, 
        embed_dim=768,
        depth=12,
        input_patch_size=16,
        activation='sigmoid', 
        name='TransUNet'
    )
    model.compile(optimizer=Adam(learning_rate=1e-4), loss=hybrid_loss, metrics=METRICS)
    return model

def build_swin_unet():
    # Pure Hierarchical Transformer Encoder Model
    model = models.swin_unet_2d(
        INPUT_SHAPE, 
        filter_num=[128, 256, 512, 1024], 
        n_labels=N_CLASSES, 
        stack_num_down=2, 
        stack_num_up=2,
        patch_size=(4, 4),
        embed_dim=128,
        num_mlp=512,
        activation='sigmoid',
        name='SwinUNet'
    )
    model.compile(optimizer=Adam(learning_rate=1e-4), loss=hybrid_loss, metrics=METRICS)
    return model

In [ ]:
# --- Cell 5: EXECUTE ALL MODELS AND COMPARE ---

# List of model builders and their names
model_experiments = [
    (build_unet, "U-Net_CNN"),
    (build_att_unet, "Attention_U-Net_CNN"),
    (build_unet_plus, "U-Net++_CNN"),
    (build_trans_unet, "TransUNet_Hybrid"),
    (build_swin_unet, "Swin-UNet_Transformer")
]

all_results = []
trained_models = {}

for builder, name in model_experiments:
    try:
        results, model, history = train_and_evaluate_model(builder, name, epochs=50) 
        all_results.append(results)
        trained_models[name] = model # Store models if needed for visualization
    except Exception as e:
        print(f"ERROR: Model {name} failed to train. Error: {e}")
        # This is often due to GPU OOM (Out of Memory) for larger models.
        # If this happens, try reducing BATCH_SIZE in Cell 1.
        all_results.append({'Model': name, 'Loss (Hybrid)': np.nan, 'IoU': np.nan, 'Dice Coefficient': np.nan, 
                            'Sensitivity (Recall)': np.nan, 'Specificity': np.nan, 'Accuracy': np.nan})

# Convert all results to a DataFrame
df_results = pd.DataFrame(all_results)
if 'Model' in df_results.columns:
    df_results.set_index('Model', inplace=True)

# --- Final Output ---
print("\n" + "="*80)
print("                     🚀 FINAL SEGMENTATION MODEL COMPARISON (ResNet34 Backbone) 🚀")
print("="*80)

# Sort by the most critical metric (Dice Coefficient)
df_results_sorted = df_results.sort_values(by='Dice Coefficient', ascending=False)
print(df_results_sorted.to_markdown(floatfmt=".4f"))

# Optional: Save results
df_results_sorted.to_csv('segmentation_comparison_final_results.csv')